In [ ]:
import json
import requests
import pandas as pd
from urllib.parse import urljoin
import time

In [ ]:
SEARCH_URL = "https://wuzzuf.net/api/search/job"
DETAILS_URL = "https://wuzzuf.net/api/job"
session = requests.Session()

In [ ]:
HEADERS = {
  'accept': 'application/vnd.api+json',
  'accept-language': 'en',
  'baggage': 'sentry-environment=production,sentry-release=426d59202416e35a74c2846bae9810f1681a4160,sentry-public_key=2529249e1d7c49d8b1990e2cd3af11ac,sentry-trace_id=51fc68ddea9e4170a8aae54bcd530181,sentry-sampled=false,sentry-sample_rand=0.8773275158181258,sentry-sample_rate=0',
  'content-type': 'application/vnd.api+json',
  'pragma': 'no-cache',
  'priority': 'u=1, i',
  'referer': 'https://wuzzuf.net/search/jobs?start=1',
  'sec-fetch-mode': 'cors',
  'sentry-trace': '51fc68ddea9e4170a8aae54bcd530181-b705242cd2a279e3-0',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36'
}




def fetch_job_summaries(start_index=0, page_size=500):

    payload = {
        "startIndex": start_index,
        "pageSize": page_size,
        "longitude": "0",
        "latitude": "0",
        "query": "",
        "searchFilters": {}
    }

    response = session.post(
        SEARCH_URL, headers=HEADERS, json=payload)
    response.raise_for_status()

    return response.json()["data"]

In [ ]:
def extract_job_ids(jobs):
    return [job["id"] for job in jobs]

In [ ]:
details_header = {
  'accept': 'application/vnd.api+json',
  'accept-language': 'en',
  'referer': 'https://wuzzuf.net/search/jobs?start=0',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36'
}
def fetch_job_details(job_ids):
  job_details = []
  batch_size = 220

  for i in range(0, len(job_ids), batch_size):
    batch = ','.join(job_ids[i:i+batch_size])
    details_url = f"{DETAILS_URL}?filter[other][ids]={batch}"
    response = session.get(
        details_url,
        headers=details_header)

    response.raise_for_status()

    data = response.json()['data']
    job_details.extend(data)

  return job_details


In [ ]:
def clean_job_details(job_details):
  records = []
  for job in job_details:
    attrs = job['attributes']
    location = attrs.get('location')
    city_obj = location.get('city')
    area_obj = location.get('area') or {}
    country_obj = location.get('country')
    workplace_obj = attrs.get('workplaceArrangement')or {}
    career_level_obj = attrs.get('careerLevel') or {}
    career_level_translations = career_level_obj.get('translations') or {}
    career_level_name = career_level_translations.get('name') or {}
    Salary = attrs.get('salary') or {}
    currency = Salary.get('currency') or {}
    period = Salary.get('period') or {}
    cnadidate = attrs.get('candidatePreferences') or {}
    educationLevel = cnadidate.get('educationLevel') or {}
    link = attrs.get('uri')
    records.append({
        "title": attrs.get("title"),
        "country": country_obj.get("name"),
        "city": city_obj.get("name"),
        "area": area_obj.get("name"),
        "careerLevel": career_level_name.get("en"),
        "workplaceArrangement": workplace_obj.get("displayedName"),
        "keywords": ", ".join(k["name"] for k in (attrs.get("keywords") or [])),
        "workExperienceYears": attrs.get("workExperienceYears"),
        "vacancies": attrs.get("vacancies"),
        "postedAt": attrs.get("postedAt"),
        "workTypes": ", ".join(w["name"] for w in (attrs.get("workTypes") or [])),
        "MiniSalary": Salary.get("min"),
        "MaxSalary": Salary.get("max"),
        "currency": currency.get("code"),
        "period":period.get('name'),
        "educationLevel":educationLevel.get('name'),
        "job link":urljoin('https://wuzzuf.net/', link)
    })
  return records

In [ ]:
def main():

    print("Fetching job summaries...")

    all_jobs = fetch_job_summaries()

    print(f"Found {len(all_jobs)} jobs")

    job_ids = extract_job_ids(all_jobs)

    print("Fetching job details...")

    job_details = fetch_job_details(job_ids)

    print(f"Fetched {len(job_details)} job details")

    jobs = clean_job_details(job_details)

    df = pd.DataFrame(jobs)

    output_filename = "jobs.xlsx"
    df.to_excel(output_filename, index=False)
    print(f"🎊The file was saved successfully: {output_filename}")

    return df


In [ ]:
if __name__ == "__main__":
    jobs = main()


In [ ]:
jobs.head()